# 04 Statistical Analysis & Forecasting

**Purpose:** Apply statistical testing to validate observations from EDA, use machine learning (Linear Regression) to forecast future revenue, and compute the final business KPIs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'retail_cleaned.csv'

df = pd.read_csv(DATA_PATH)
print(f"Dataset Shape: {df.shape}")

### Step 1: Correlation Matrix for Numerical Columns
**Why:** Identify linear relationships between numeric features, such as how `quantity` or `discount_applied` correlates with `total_spent`.

In [ ]:
num_cols = ['price_per_unit', 'quantity', 'total_spent', 'discount_applied']
corr = df[num_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Matrix')
plt.show()

### Step 2: Hypothesis Testing (Welch's T-Test)
**Question:** Is the Average Order Value (AOV) significantly different between discounted vs. non-discounted transactions?

**Why:** EDA showed a slight difference in AOV. We must use a t-test to determine if this difference is statistically significant (p < 0.05) or just random noise.

In [ ]:
discounted = df[df['discount_applied'] == 1]['total_spent']
no_discount = df[df['discount_applied'] == 0]['total_spent']

t_stat, p_value = stats.ttest_ind(discounted, no_discount, equal_var=False)

print(f"Discounted AOV: ${discounted.mean():.2f}")
print(f"No Discount AOV: ${no_discount.mean():.2f}")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.6f}")

if p_value < 0.05:
    print("\nConclusion: REJECT the null hypothesis. The difference in AOV is statistically significant.")
else:
    print("\nConclusion: FAIL TO REJECT the null hypothesis. The difference in AOV is not statistically significant.")

### Step 3: Simple Linear Regression & Forecasting
**Why:** We want to model historical monthly revenue and forecast performance for the next 2 quarters (6 months) to aid financial planning.

In [ ]:
# Aggregate revenue by month
monthly_rev = df.groupby(['transaction_year', 'transaction_month'])['total_spent'].sum().reset_index()
monthly_rev = monthly_rev.sort_values(['transaction_year', 'transaction_month']).reset_index(drop=True)

# Create an ordinal time index (0, 1, 2... N)
monthly_rev['time_index'] = monthly_rev.index

# Fit Linear Regression Model
X = monthly_rev[['time_index']]
y = monthly_rev['total_spent']
model = LinearRegression()
model.fit(X, y)

# Forecast next 6 months (2 quarters)
future_indices = np.array(range(len(monthly_rev), len(monthly_rev) + 6)).reshape(-1, 1)
forecast_values = model.predict(future_indices)

print("Model R² Score:", model.score(X, y).round(4))

# Visualize Historical + Forecasted Revenue
plt.figure(figsize=(10, 5))
plt.plot(monthly_rev['time_index'], y, marker='o', label='Historical Revenue', color='#1f77b4')
plt.plot(monthly_rev['time_index'], model.predict(X), color='black', linestyle='--', label='Trend Line')
plt.plot(future_indices, forecast_values, marker='s', color='#ff7f0e', label='Forecast (Next 6 Months)')
plt.title('Monthly Revenue Forecast using Linear Regression')
plt.xlabel('Months Since Start of Dataset')
plt.ylabel('Total Revenue ($)')
plt.legend()
plt.show()

### Step 4: Compute Business KPIs
**Why:** Extracting the exact top-level metrics needed for the executive Tableau dashboard summary.

In [ ]:
total_revenue = df['total_spent'].sum()
aov = df['total_spent'].mean()
discount_rate = (df['discount_applied'].sum() / len(df)) * 100
online_pct = (len(df[df['location'] == 'Online']) / len(df)) * 100
instore_pct = (len(df[df['location'] == 'In-Store']) / len(df)) * 100

print("="*40)
print("         FINAL BUSINESS KPIs")
print("="*40)
print(f"Total Revenue    : ${total_revenue:,.2f}")
print(f"Average Order Vol: ${aov:,.2f}")
print(f"Discount Rate    : {discount_rate:.2f}%")
print(f"Online Sales     : {online_pct:.2f}%")
print(f"In-Store Sales   : {instore_pct:.2f}%")
print("="*40)